# Prompt caching for open-weight LLMs on Amazon SageMaker AI (SGLang DLC)

This notebook deploys **DeepSeek-R1-Distill-Qwen-1.5B and -7B** to Amazon SageMaker AI real-time endpoints
using the **AWS SGLang Deep Learning Container** (SageMaker variant) and measures what SGLang's
RadixAttention prefix cache buys you:

1. **Prefix-length sweep** – cold (cache miss) vs. warm (cache hit) TTFT and end-to-end latency for shared
   prefixes of 512–8,192 **tokens**. Every warm request keeps the prefix but sends a **different user query**.
2. **Load test** – sustained throughput and p50/p95 latency at 1–32 concurrent clients, cache **on vs. off**.
   This is what drives cost on SageMaker, which bills per instance-hour, not per request.
3. **Eviction test** – what happens to the hit rate when the distinct prefixes no longer fit in the KV pool.
4. **Cost views** – $/1M requests derived from measured throughput, and a break-even view against
   Amazon Bedrock prompt caching (modelled, not like-for-like).

All logic lives in [`run_benchmark.py`](run_benchmark.py); this notebook calls it step by step.

**Where to run:** SageMaker Studio JupyterLab or a notebook instance **in the same Region as the endpoints**
(a CPU kernel such as `ml.t3.medium` is enough – the GPU work happens on the endpoints). TTFT is measured
client-side through the SageMaker runtime API, so a client far from the Region adds network latency.

**Cost:** each endpoint is one `ml.g5.xlarge` (1x A10G 24 GB, $1.408/hr in us-east-1 at time of writing).
The full run (4 endpoints, ~45 min including deployment) costs roughly $4–5. **Run the clean-up cell at the end.**

**Prerequisites:** a SageMaker execution role, and a quota of 4 x `ml.g5.xlarge for endpoint usage`
(or set `SIZES = ["1.5b"]` / `RUN_CONTROL = False` below to use fewer).

In [ ]:
%pip install -q "boto3>=1.35" "sagemaker>=2.240,<3" tokenizers==0.22.2 huggingface_hub pandas matplotlib

In [ ]:
import os, json, boto3
import pandas as pd
from IPython.display import Image, display
import run_benchmark as rb

try:
    import sagemaker
    ROLE = os.environ.get("SM_EXEC_ROLE") or sagemaker.get_execution_role()
except Exception:
    ROLE = os.environ.get("SM_EXEC_ROLE", "arn:aws:iam::<ACCOUNT_ID>:role/<SageMakerExecutionRole>")

SIZES = ["1.5b", "7b"]        # models to benchmark (both fit on one A10G)
RUN_CONTROL = True            # also deploy a cache-OFF endpoint per model (negative control)
SUFFIX = "nb"                 # endpoint names: pc-<size>-cache-<suffix>, pc-<size>-nocache-<suffix>

print("region :", rb.REGION)
print("role   :", ROLE)
print("image  :", rb.IMAGE_URI)
print("models :", {s: rb.MODELS[s] for s in SIZES})

## 1. Deploy SGLang endpoints

The SageMaker variant of the SGLang DLC turns every `SM_SGLANG_*` environment variable into an SGLang server flag
(`SM_SGLANG_MEM_FRACTION_STATIC=0.85` becomes `--mem-fraction-static 0.85`). `rb.deploy()` sets:

| Variable | Why |
|---|---|
| `SM_SGLANG_MODEL_PATH` | Hugging Face model ID |
| `SM_SGLANG_ENABLE_CACHE_REPORT=true` | return `usage.prompt_tokens_details.cached_tokens` so we can **prove** a hit |
| `SM_SGLANG_MEM_FRACTION_STATIC=0.85`, `SM_SGLANG_CONTEXT_LENGTH=16384` | memory budget for weights + KV pool, max sequence |
| `SM_SGLANG_DISABLE_RADIX_CACHE=true` | **control endpoint only** – turns prefix caching off |

RadixAttention is **on by default**; nothing is needed to enable it. GPU instances need
`InferenceAmiVersion=al2023-ami-sagemaker-inference-gpu-4-1` for the CUDA 13 image. Deployment takes ~8–10 minutes.

In [ ]:
endpoints = {}
for s in SIZES:
    for cache in ([True, False] if RUN_CONTROL else [True]):
        label = f"{s}-{'cache' if cache else 'nocache'}"
        endpoints[label] = rb.deploy(s, cache, ROLE, SUFFIX, wait=False)

sm = rb.sm_client()
for label, ep in endpoints.items():
    sm.get_waiter("endpoint_in_service").wait(EndpointName=ep, WaiterConfig={"Delay": 30, "MaxAttempts": 80})
    print("InService:", label, ep)

## 2. How much GPU memory is left for the KV cache?

SGLang logs the KV pool size at start-up (`KV Cache is allocated ... #tokens` / `max_total_num_tokens`).
We read it from CloudWatch Logs. This pool holds **both** cached prefixes and the KV of in-flight requests;
when it fills, the radix cache evicts the least-recently-used prefixes.

In [ ]:
os.makedirs(rb.RESULTS_DIR, exist_ok=True)
kv = {label: rb.kv_pool(ep) for label, ep in endpoints.items()}
json.dump(kv, open(f"{rb.RESULTS_DIR}/kv_pool.json", "w"), indent=2)
pd.DataFrame(kv).T[["weights_gb", "avail_mem_gb", "kv_tokens", "kv_k_gb", "kv_v_gb", "max_running_requests"]]

## 3. Sanity check: same prefix, different user queries

The prefix is built with the model's tokenizer so that `BOS + system prompt + <｜User｜>` is **exactly** N tokens.
A random nonce at the start of the system prompt guarantees the first request is a miss (SageMaker does not expose
SGLang's `/flush_cache`). The next requests reuse the prefix with a new query each time: `cached_tokens` should be
≈ N (the prefix), **not** the whole prompt.

In [ ]:
import random, uuid
rt, rng = rb.rt_client(), random.Random(0)
s = SIZES[0]
system = rb.build_system_prompt(2048, uuid.uuid4().hex[:16])
for i in range(4):
    for label in [f"{s}-cache", f"{s}-nocache"]:
        if label in endpoints:
            r = rb.invoke_stream(rt, endpoints[label], rb.MODELS[s], system, rb.make_query(rng), 32)
            print(f"{label:<12} req {i}: TTFT {r['ttft_ms']:>7} ms  e2e {r['e2e_ms']:>7} ms  "
                  f"cached {r['cached_tokens']}/{r['prompt_tokens']}")

## 4. Prefix-length sweep (concurrency 1)

For each prefix length in `rb.SWEEP_PREFIX_TOKENS` we build `rb.SWEEP_TRIALS` independent prefixes. Each trial sends
1 cold request and `rb.SWEEP_WARM_PER_TRIAL` warm requests with **different queries**, all with a fixed 32-token
decode (`ignore_eos`). We report medians across trials for both **TTFT** and **end-to-end latency** (prefill + decode).
Running the same sweep on the cache-off endpoint is the negative control: warm ≈ cold and `cached_tokens` = 0.

In [ ]:
for label, ep in endpoints.items():
    rb.bench(ep, label.split("-")[0], label, ["sweep"])

## 5. Throughput under load (cache on vs. off)

Closed loop: C clients each send back-to-back requests. Each request picks one of `rb.LOAD_NUM_PREFIXES` 2,048-token
system prompts (think: four agents or tenants) and a fresh user query, and generates 128 tokens. Sustained
requests/s per instance is what turns into cost on an instance-hour-billed endpoint.

In [ ]:
for label, ep in endpoints.items():
    rb.bench(ep, label.split("-")[0], label, ["load"])

## 6. Eviction: more distinct prefixes than the KV pool can hold

Round-robin over N distinct 8,192-token prefixes for three passes, and measure the hit rate on passes 2–3. While
N × 8,192 fits in the pool every prefix stays resident; once it doesn't, LRU evicts each prefix just before it comes
round again and the hit rate collapses. (Round-robin is the worst case for LRU – real traffic with a few hot prefixes
degrades more gracefully, and `--radix-eviction-policy lfu|slru` can help.)

In [ ]:
for label, ep in endpoints.items():
    if label.endswith("-cache"):
        rb.bench(ep, label.split("-")[0], label, ["evict"])

## 7. Tables and figures

In [ ]:
rb.analyze()
sweep = pd.read_csv(f"{rb.RESULTS_DIR}/table_sweep.csv")
sweep[sweep.cache][["model", "prefix_tokens", "ttft_cold_ms", "ttft_warm_ms", "ttft_reduction_pct",
                    "e2e_cold_ms", "e2e_warm_ms", "e2e_reduction_pct", "cached_tokens_warm_median", "prompt_tokens_median"]]

In [ ]:
for f in ["fig1_reduction_vs_prefix.png", "fig2_ttft_cold_warm.png", "fig3_throughput.png", "fig4_eviction.png"]:
    p = f"{rb.RESULTS_DIR}/{f}"
    if os.path.exists(p):
        display(Image(p))

## 8. Cost: measured throughput, not modelled GPU-seconds

SageMaker bills the instance by the hour whether it is busy or idle. So the honest cost per request is
`instance $/hr ÷ sustained requests per hour`, measured at the concurrency (and latency) you are willing to run.
`usd_per_1M_req_at_full_util` assumes the instance is kept busy at that concurrency; at lower utilisation divide by
your utilisation.

In [ ]:
load = pd.read_csv(f"{rb.RESULTS_DIR}/table_load.csv")
load[["model", "cache", "concurrency", "req_per_s", "ttft_p50_ms", "ttft_p95_ms", "e2e_p50_ms", "e2e_p95_ms",
      "hit_rate_pct", "usd_per_1M_req_at_full_util"]]

### Break-even against Amazon Bedrock prompt caching (modelled)

Not a like-for-like benchmark – Bedrock prompt caching supports a specific set of models, and DeepSeek-R1-Distill
is not one of them. The question it answers: *at what request volume does one always-on SGLang instance cost less
than paying per token on Bedrock for the same request shape?* Bedrock prices are in `rb.BEDROCK`
(us-east-1 on-demand at time of writing – check the Amazon Bedrock pricing page).

In [ ]:
pd.read_csv(f"{rb.RESULTS_DIR}/table_bedrock_breakeven.csv")

## 9. Clean up

Deletes every endpoint, endpoint config, and model whose name starts with `pc-` **and** carries the tag
`project=prompt-cache-blog` (so nothing else in your account is touched).

In [ ]:
rb.cleanup()